Preparation

In [ ]:
#Setup - Import libraries and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

#Settings
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 6]

#Load data
from google.colab import drive
drive.mount('/content/drive')
Data_Path = "/content/drive/MyDrive/Mid-term project/Data/"
bug_data = pd.read_csv(Data_Path + "bug_dataset_with_priority_and_severity.csv")



MessageError: Error: credential propagation was unsuccessful

Preprocessing


In [ ]:
#Replace string with int ratings
bug_data["severity"] = bug_data["severity"].replace({"Low": 1, "Medium": 2, "High": 3, "Critical": 4})
bug_data["priority"] = bug_data["priority"].replace({"Low": 1, "Normal": 2, "High": 3, "Urgent": 4})

#Recode severity
def calculate_severity(row):
    score = 0
    # Environment
    if row["environment"] == "Production":
        score += 2
    elif row["environment"] == "Staging":
        score += 1
    # Bug Category
    if row["bug_category"] == "Security Vulnerability":
        score += 4
    elif row["bug_category"] in [
        "Memory Leak",
        "Database Bug",
        "Concurrency Bug",
        "Authentication Bug",
        "Authorization Bug"
    ]:
        score += 3
    elif row["bug_category"] in [
        "Backend Logic Bug",
        "Performance Bug",
        "API Bug",
        "Deployment Bug",
        "Cloud Configuration Bug"
    ]:
        score += 2
    else:
        score += 1
    # Error Code
    if row["error_code"] >= 500:
        score += 2
    elif row["error_code"] >= 400:
        score += 1
    return score

#Apply recode
bug_data["severity_score"] = bug_data.apply(
    calculate_severity,
    axis=1
)

def severity_class(score):
    if score <= 2:
        return 1      # Low
    elif score <= 4:
        return 2      # Medium
    elif score <= 6:
        return 3      # High
    else:
        return 4      # Critical

bug_data["severity"] = bug_data["severity_score"].apply(severity_class)

#Recode priority
def priority_class(row):
    score = row["severity"]
    if row["environment"] == "Production":
        score += 1
    if score >= 5:
        return 4      # P1 Critical
    elif score == 4:
        return 3      # P2 High
    elif score == 3:
        return 2      # P3 Normal
    else:
        return 1      # P4 Low

#Apply recode
bug_data["priority"] = bug_data.apply(
    priority_class,
    axis=1
)

In [ ]:
#Check for missing values
if bug_data.isna().values.any() == True:
  print("There are missing values in the dataset")
  print(bug_data.isna().sum())
else:
  print("There are no missing values in the dataset")

#Replace missing values with "Unknown"
bug_data_filled = bug_data.fillna("Unknown")

#Check
print(bug_data_filled.isna().sum())


In [ ]:
#Check for duplicate values
if bug_data_filled.duplicated().any() == True:
  print("There are duplicate values in the dataset")
  print(bug_data_filled.duplicated().sum())
else:
  print("There are no duplicate values in the dataset")

Basic Statistics

In [ ]:
#Number of bugs
print("Number of bugs: ", len(bug_data_filled))

#Number of errors per error_code
print("Number of errors per error_code: \n", bug_data_filled["error_code"].value_counts())

#Number of errors per severity
print("Number of errors per severity: \n", bug_data_filled["severity"].value_counts())

#Number of errors per priority rating
print("Number of errors per priority rating: \n", bug_data_filled["priority"].value_counts())

#Number of errors per environment
print("Number of errors per environment: \n", bug_data_filled["environment"].value_counts())

In [ ]:
#Severity by bug category
print("Priority by bug category: \n", bug_data_filled.groupby("bug_category")["priority"].mean())

#Priority by bug category
print("Severity by bug category: \n", bug_data_filled.groupby("bug_category")["severity"].mean())

#Severity by environment
environment_severity = bug_data_filled.groupby("environment")["severity"].agg
print("Severity by environment: \n", bug_data_filled.groupby("environment")["severity"].mean())

#Priority by environment
print("Priority by environment: \n", bug_data_filled.groupby("environment")["priority"].mean())

#Severity by bug domain
print("Severity by bug domain: \n", bug_data_filled.groupby("bug_domain")["severity"].mean())

#Priority by bug domain
print("Priority by bug domain: \n", bug_data_filled.groupby("bug_domain")["priority"].mean())

Visualisation

In [ ]:
#Bar Plot of Severity based on Bug Category

severity_by_domain = bug_data_filled.groupby("bug_category")["severity"].mean().sort_values(ascending=True)

plt.figure(figsize=(12, 6))
sns.barplot(x=severity_by_domain.index, y=severity_by_domain.values, palette='viridis')
plt.title('Mean Severity Rating by Bug Category')
plt.xlabel('Bug Domain')
plt.ylabel('Mean Severity Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Bar Plot of Severity by environment

severity_by_environment = bug_data_filled.groupby("environment")["severity"].median().sort_values(ascending = True)

plt.figure(figsize=(12, 6))
sns.barplot(x=severity_by_environment.index, y=severity_by_environment.values, palette='viridis')
plt.title('Median Severity Rating by Environment')
plt.xlabel('Environment')
plt.ylabel('Median Severity Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Bar Plot of Priority by environment

priority_by_environment = bug_data_filled.groupby("environment")["priority"].median().sort_values(ascending = True)

plt.figure(figsize=(12, 6))
sns.barplot(x=priority_by_environment.index, y=priority_by_environment.values, palette='viridis')
plt.title('Median Priority Rating by Environment')
plt.xlabel('Environment')
plt.ylabel('Median Priority Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Scatterplot to check correlation between priority and severity ratings
fig, ax = plt.subplots(figsize = (12, 6))
ax.scatter(bug_data_filled["priority"], bug_data_filled["severity"])
sns.regplot(x = bug_data_filled['priority'], y = bug_data_filled['severity'])
plt.suptitle('Correlation Between Severity and Priority', y=1.02)
plt.xlabel('Priority Rating')
plt.ylabel('Severity Rating')
plt.tight_layout()
plt.show()

#Check correlation coefficient
correlation = bug_data_filled['priority'].corr(bug_data_filled['severity'])
print("Correlation between priority and severity:", correlation)

Model Preparation

In [ ]:
#Combined severity/priority rating
bug_data_filled["severity_priority"] = (bug_data_filled["severity"] + bug_data_filled["priority"])//2

In [ ]:
#Create text column
bug_data_filled["text"] = (
    bug_data_filled["title"].fillna("") +
    " " +
    bug_data_filled["description"].fillna("")
)

#Define categorical features
x = bug_data_filled[
    [
        "text",
        "error_code",
        "bug_domain",
        "bug_category",
        "environment",
        "tech_stack"
    ]
]

#Define y as combination of severity and priority scores
y = bug_data_filled["severity_priority"]

print(bug_data_filled["severity_priority"].value_counts())
print(bug_data_filled.shape)


In [ ]:
#Test-Train Split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2, #80/20 split
    random_state = 42,
    stratify = y
)

In [ ]:
#Create processing pipeline
preprocessor = ColumnTransformer(
    transformers = [
        (
          "text",
         TfidfVectorizer(
             stop_words = 'english', #Remove common words
             max_features = 3000, #Keeps 3000 most informative words
             ngram_range = (1, 2), #Uses single and two-word phrases
             min_df = 2 #Ignores words that only appear once)
         ),
         "text"
        ),

        (
            "categorical",
            #Encode features as one numeric array
            OneHotEncoder(handle_unknown = "ignore"),
            [
                "text",
                "bug_domain",
                "bug_category",
                "environment",
                "tech_stack",
            ]
        )
    ]
)

Logistic Regression:

In [ ]:
#Logistic Regression
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter = 1000))
])

#Train Model
lr_pipeline.fit(x_train, y_train)

#Test Model
lr_predictions = lr_pipeline.predict(x_test)

#Evaluate Model
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, lr_predictions))
print(classification_report(y_test, lr_predictions))
print(confusion_matrix(y_test, lr_predictions))

Random Forest:

In [ ]:
#Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators = 200,
        random_state = 42,
        max_depth = None
    ))
])

#Train
rf_pipeline.fit(x_train, y_train)

#Test
rf_predictions = rf_pipeline.predict(x_test)

#Evaluate
print("Random Forest")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions))
print(confusion_matrix(y_test, rf_predictions))
#

In [ ]:
#Cross Validation of Logistic Regression
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    lr_pipeline,
    x,
    y,
    cv = 5,
    scoring = "accuracy"
)

#Print cross validation scores
print("Cross-validation accuracy:", scores)
print("Mean accuracy:", scores.mean())